In [ ]:
import os

# Am schimbat calea să fie totul cu litere mici
input_path = '/kaggle/input/datasets/chrisfilo/urbansound8k'

# Verificăm dacă folderul chiar există înainte de a-l parcurge
if os.path.exists(input_path):
    print("Succes! Folderul a fost găsit.")
    for dirname, _, filenames in os.walk(input_path):
        for filename in filenames[:5]:
            print(os.path.join(dirname, filename))
else:
    print(f"Eroare: Calea {input_path} nu este corectă. Verifică numele folderului în dreapta!")

In [3]:
" Model de CNN facut de la 0 pe spectograma"

import os
import random
import torch
import torchaudio
import pandas as pd
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torchaudio.transforms as T
import numpy as np
import matplotlib.pyplot as plt

class AudioAugment:
    def add_noise(self, wav):
        # aplicat DUPA rms_normalize → noise_level consistent (1% din energie)
        noise_level = torch.rand(1).item() * 0.01
        noise = torch.randn_like(wav) * noise_level
        return wav + noise

    def time_shift(self, wav):
        max_shift = int(0.2 * wav.shape[-1])
        shift = torch.randint(-max_shift, max_shift, (1,)).item()
        return torch.roll(wav, shifts=shift, dims=-1)

    def change_volume(self, wav):
        # range redus: [0.7, 1.3] in loc de [0.5, 1.5]
        gain = torch.rand(1).item() * 0.6 + 0.7
        return wav * gain

    def random_crop(self, wav):
        length = wav.shape[-1]
        crop_size = int(0.8 * length)
        if length <= crop_size:
            return wav
        start = torch.randint(0, length - crop_size, (1,)).item()
        cropped = wav[..., start:start + crop_size]
        # pad prin repetitie in loc de zerouri — evita silentiu artificial
        repeats = (length // crop_size) + 2
        repeated = cropped.repeat(1, repeats)
        return repeated[..., :length]


augment = AudioAugment()

# augmentari aplicate INAINTE de rms_normalize
PRE_NORMALIZE_AUGMENTATIONS = [
    lambda wav, sr: augment.time_shift(wav),
    lambda wav, sr: augment.change_volume(wav),
    lambda wav, sr: augment.random_crop(wav),
]


class AudioDataset(Dataset):
    def __init__(self, df, audio_dir, sr=16000, n_mels=128,
                 duration=4.0, apply_augment=False):
        self.df             = df.reset_index(drop=True)
        self.audio_dir      = audio_dir
        self.sr             = sr
        self.target_length  = int(sr * duration)
        self.apply_augment  = apply_augment
        self.labels         = sorted(self.df["classID"].unique())
        self.label2id       = {l: i for i, l in enumerate(self.labels)}

    def __len__(self):
        return len(self.df)

    def load_audio(self, path, target_length=None):
        wav, sr = torchaudio.load(path)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        if sr != self.sr:
            wav = torchaudio.functional.resample(wav, sr, self.sr)
        if target_length is not None:
            if wav.shape[-1] >= target_length:
                wav = wav[..., :target_length]
            else:
                wav = torch.nn.functional.pad(wav, (0, target_length - wav.shape[-1]))
        return wav

    def rms_normalize(self, wav):
        # ignora zerourile de padding la calculul RMS
        mask = wav.abs() > 1e-8
        if mask.sum() == 0:
            return wav
        rms = torch.sqrt((wav[mask] ** 2).mean())
        if rms < 1e-4:
            return wav
        return wav / (rms + 1e-9)

    def per_clip_normalize(self, mel):
        return (mel - mel.mean()) / (mel.std() + 1e-6)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = os.path.join(self.audio_dir, f"fold{row['fold']}", row["slice_file_name"])
        label = row["classID"]

        wav = self.load_audio(path, target_length=self.target_length)

        if self.apply_augment:
            aug_fn = random.choice(PRE_NORMALIZE_AUGMENTATIONS + [None])
            if aug_fn is not None:
                wav = aug_fn(wav, self.sr)

        # 2. rms_normalize dupa transformari, inainte de zgomot
        wav = self.rms_normalize(wav)

        if self.apply_augment:
            # 3. zgomot aditiv — dupa normalizare, nivel consistent
            if random.random() < 0.3:
                wav = augment.add_noise(wav)

        return wav, self.label2id[label]

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1)
        self.bn1   = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(out_channels)

        self.dropout = nn.Dropout2d(dropout)

        # shortcut (dacă schimbăm dimensiunea)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.dropout(out)

        out += identity
        out = F.relu(out)
        return out


class CNNClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.block1 = ResidualBlock(1, 32, stride=2, dropout=0.1)  # 0.05 (inceput)   best(0.1)
        self.block2 = ResidualBlock(32, 64, stride=2, dropout=0.2)   # 0.1            best(0.2)
        self.block3 = ResidualBlock(64, 128, stride=2, dropout=0.3) # 0.15           best(0.3)
        self.block4 = ResidualBlock(128, 256, stride=2, dropout=0.4) # 0.2           best(0.4)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),  # 0.4        best(0.6)
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        x = self.pool(x)
        x = x.view(x.size(0), -1)

        return self.classifier(x)
print("Aici")

In [4]:

DATA_PATH = "/kaggle/input/datasets/chrisfilo/urbansound8k"
CSV_PATH = os.path.join(DATA_PATH, "UrbanSound8K.csv")
AUDIO_DIR = DATA_PATH

BATCH_SIZE = 32    #64
EPOCHS = 40
LR = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import copy   

MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_mels=128,
    n_fft=1024,
    hop_length=256,
).to(DEVICE)

DB_TRANSFORM = torchaudio.transforms.AmplitudeToDB().to(DEVICE)

TIME_MASK = T.TimeMasking(time_mask_param=30)
FREQ_MASK = T.FrequencyMasking(freq_mask_param=15)

def apply_spec_augment(mel):
    if random.random() < 0.8:
        mel = TIME_MASK(mel)

    if random.random() < 0.8:
        mel = FREQ_MASK(mel)

    return mel

def wav_to_mel(wav, apply_augment=False):
    mel = MEL_TRANSFORM(wav)
    mel = DB_TRANSFORM(mel)

    if apply_augment:
        mel = apply_spec_augment(mel)

    mean = mel.mean(dim=(-2, -1), keepdim=True)
    std  = mel.std(dim=(-2, -1), keepdim=True)

    return (mel - mean) / (std + 1e-6)

class EarlyStopping:
    def __init__(self, patience=4, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.best_model_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_model_state = copy.deepcopy(model.state_dict())
            return False
        else:
            self.counter += 1

        return self.counter >= self.patience

def train_one_epoch(model, loader, optimizer, criterion, scheduler):
    model.train()
    total_loss = 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        x = wav_to_mel(x, apply_augment = True)  
        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        scheduler.step() 

        total_loss += loss.item()

    return total_loss / len(loader)

def validate(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            x = wav_to_mel(x)
            
            outputs = model(x)
            loss = criterion(outputs, y)
            total_loss += loss.item()

    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            x = wav_to_mel(x)

            outputs = model(x)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

def run_cross_validation():
    df = pd.read_csv(CSV_PATH)
    all_accuracies = []

    for fold in range(1, 11):
        print(f"\n========== FOLD {fold} ==========")
        # split folds
        train_df_full = df[df["fold"] != fold]
        test_df = df[df["fold"] == fold]

        # train/val split
        train_df, val_df = train_test_split(
            train_df_full,
            test_size=0.1,
            stratify=train_df_full["classID"],
            random_state=42
        )

        # datasets
        train_dataset = AudioDataset(train_df, AUDIO_DIR, apply_augment=True)
        val_dataset   = AudioDataset(val_df, AUDIO_DIR, apply_augment=False)
        test_dataset  = AudioDataset(test_df, AUDIO_DIR, apply_augment=False)

        # loaders
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers = 4, pin_memory=True)
        val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers =4, pin_memory=True)
        test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

        # model
        model = CNNClassifier().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        
        steps_per_epoch = len(train_loader)

    
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=3e-3,
        epochs=EPOCHS,
        steps_per_epoch=steps_per_epoch,
        pct_start=0.3,
        anneal_strategy='cos'
        )


        '''
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=3)  #  initial(2)
        '''
        criterion = nn.CrossEntropyLoss(label_smoothing = 0.05)  #label smoothing - explica in paper de ce
        
        early_stopper = EarlyStopping(patience=6)  # era 4 
        
        # training loop
        for epoch in range(EPOCHS):
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scheduler)
            val_loss   = validate(model, val_loader, criterion)
            #scheduler.step(val_loss)
            print(f"Epoch {epoch+1}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

            if early_stopper.step(val_loss, model):
                print("Early stopping triggered")
                break

        #restore best model
        model.load_state_dict(early_stopper.best_model_state)
        # test
        acc = evaluate(model, test_loader)
        print(f"Fold {fold} Accuracy: {acc:.4f}")

        all_accuracies.append(acc)

    mean_acc = np.mean(all_accuracies)
    std_acc = np.std(all_accuracies)   

    print("\n========== FINAL RESULTS ==========")
    print(f"Mean Accuracy: {mean_acc:.4f}")
    print(f"Std Dev: {std_acc:.4f}")

    # boxplot
    plt.boxplot(all_accuracies)
    plt.title("10-Fold Cross Validation Accuracy")
    plt.ylabel("Accuracy")
    plt.show()

if __name__ == "__main__":
    run_cross_validation()


========== FOLD 1 ==========


NameError: name 'CNNClassifier' is not defined

In [ ]:
" Model de Efficient net pe spectograma"

import os
import torchvision.models as models

import torch
import torch.nn as nn
import torch.nn.functional as F

DATA_PATH = "/kaggle/input/datasets/chrisfilo/urbansound8k"
CSV_PATH = os.path.join(DATA_PATH, "UrbanSound8K.csv")
AUDIO_DIR = DATA_PATH

BATCH_SIZE = 32 
EPOCHS = 40
LR = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODELS_DIR = "./saved_models"
os.makedirs(MODELS_DIR, exist_ok=True)

import copy   

MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_mels=260,
    n_fft=1024,
    hop_length=246,
).to(DEVICE)

DB_TRANSFORM = torchaudio.transforms.AmplitudeToDB().to(DEVICE)

TIME_MASK = T.TimeMasking(time_mask_param=20)  
FREQ_MASK = T.FrequencyMasking(freq_mask_param=20)

def apply_spec_augment(mel):
    if random.random() < 0.8:
        mel = TIME_MASK(mel)
    if random.random() < 0.8:
        mel = FREQ_MASK(mel)
    return mel

def wav_to_mel(wav, apply_augment=False):
    mel = MEL_TRANSFORM(wav)
    mel = DB_TRANSFORM(mel)

    if apply_augment:
        mel = apply_spec_augment(mel)

    mean = mel.mean(dim=(-2, -1), keepdim=True)
    std  = mel.std(dim=(-2, -1), keepdim=True)
    mel = (mel - mean) / (std + 1e-6)

    mel = F.interpolate(mel, size=(224, 224), mode='bilinear', align_corners=False)
    mel = mel.repeat(1, 3, 1, 1)

    return mel

class EarlyStopping:
    def __init__(self, patience=7, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.best_model_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_model_state = copy.deepcopy(model.state_dict())
            return False
        else:
            self.counter += 1
        return self.counter >= self.patience

def build_model():
    """Construieste si returneaza un model EfficientNet-B2 proaspat."""
    model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 10)
    return model.to(DEVICE)

def get_model_path(fold):
    return os.path.join(MODELS_DIR, f"model_fold_{fold}.pt")

def train_one_epoch(model, loader, optimizer, criterion, scheduler):
    model.train()
    total_loss = 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x = wav_to_mel(x, apply_augment=True)
        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)

def validate(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            x = wav_to_mel(x)
            outputs = model(x)
            loss = criterion(outputs, y)
            total_loss += loss.item()

    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            x = wav_to_mel(x)
            outputs = model(x)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

def run_cross_validation():
    df = pd.read_csv(CSV_PATH)
    all_accuracies = []

    for fold in range(1, 11):
        print(f"\n========== FOLD {fold} ==========")

        model_path = get_model_path(fold)

        # ── Construim dataset-ul de test (necesar indiferent de caz) ────────
        test_df = df[df["fold"] == fold]
        test_dataset = AudioDataset(test_df, AUDIO_DIR, apply_augment=False)
        test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

        model = build_model()

        # ── Daca modelul exista deja, sari peste antrenare ───────────────────
        if os.path.exists(model_path):
            print(f"Model gasit la '{model_path}'. Se sare peste antrenare.")
            model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        else:
            print(f"Niciun model salvat pentru fold {fold}. Incepe antrenarea...")

            train_df_full = df[df["fold"] != fold]
            train_df, val_df = train_test_split(
                train_df_full,
                test_size=0.1,
                stratify=train_df_full["classID"],
                random_state=42
            )

            train_dataset = AudioDataset(train_df, AUDIO_DIR, apply_augment=True)
            val_dataset   = AudioDataset(val_df,   AUDIO_DIR, apply_augment=False)

            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
            val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

            optimizer = torch.optim.AdamW([
                {"params": model.features.parameters(),    "lr": LR * 0.1},
                {"params": model.classifier.parameters(),  "lr": LR},
            ], weight_decay=1e-4)

            scheduler = torch.optim.lr_scheduler.OneCycleLR(
                optimizer,
                max_lr=[LR * 0.1, LR],
                epochs=EPOCHS,
                steps_per_epoch=len(train_loader),
                pct_start=0.3,
                anneal_strategy='cos'
            )

            criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
            early_stopper = EarlyStopping(patience=7)

            for epoch in range(EPOCHS):
                train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scheduler)
                val_loss   = validate(model, val_loader, criterion)
                print(f"Epoch {epoch+1}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

                if early_stopper.step(val_loss, model):
                    print("Early stopping triggered")
                    break

            # ── Incarca cel mai bun state si salveaza pe disc ────────────────
            model.load_state_dict(early_stopper.best_model_state)
            torch.save(model.state_dict(), model_path)
            print(f"Model salvat la '{model_path}'")

        # ── Testare (rulata mereu) ───────────────────────────────────────────
        acc = evaluate(model, test_loader)
        print(f"Fold {fold} Accuracy: {acc:.4f}")
        all_accuracies.append(acc)

    mean_acc = np.mean(all_accuracies)
    std_acc  = np.std(all_accuracies)

    print("\n========== FINAL RESULTS ==========")
    print(f"Mean Accuracy: {mean_acc:.4f}")
    print(f"Std Dev: {std_acc:.4f}")

    plt.boxplot(all_accuracies)
    plt.title("10-Fold Cross Validation Accuracy")
    plt.ylabel("Accuracy")
    plt.show()

if __name__ == "__main__":
    run_cross_validation()

/usr/local/lib/python3.12/dist-packages/torchaudio/functional/functional.py:581: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (260) may be set too high. Or, the value for `n_freqs` (513) may be set too low.
  warnings.warn(



========== FOLD 1 ==========
Niciun model salvat pentru fold 1. Incepe antrenarea...
Epoch 1/40 | Train: 2.1797 | Val: 2.0864
Epoch 2/40 | Train: 1.8603 | Val: 1.7255
Epoch 3/40 | Train: 1.4428 | Val: 1.3256
Epoch 4/40 | Train: 1.1650 | Val: 1.0544
Epoch 5/40 | Train: 0.9856 | Val: 0.8944
Epoch 6/40 | Train: 0.8584 | Val: 0.8265
Epoch 7/40 | Train: 0.7890 | Val: 0.7652
Epoch 8/40 | Train: 0.7389 | Val: 0.7237
Epoch 9/40 | Train: 0.6981 | Val: 0.6953
Epoch 10/40 | Train: 0.6643 | Val: 0.6669
Epoch 11/40 | Train: 0.6514 | Val: 0.6635
Epoch 12/40 | Train: 0.6331 | Val: 0.6299
Epoch 13/40 | Train: 0.6122 | Val: 0.6285
Epoch 14/40 | Train: 0.6076 | Val: 0.6655
Epoch 15/40 | Train: 0.6002 | Val: 0.6055
Epoch 16/40 | Train: 0.5963 | Val: 0.6425
Epoch 17/40 | Train: 0.5839 | Val: 0.5962
Epoch 18/40 | Train: 0.5761 | Val: 0.6016
Epoch 19/40 | Train: 0.5739 | Val: 0.5930
Epoch 20/40 | Train: 0.5753 | Val: 0.6108
Epoch 21/40 | Train: 0.5668 | Val: 0.5996
Epoch 22/40 | Train: 0.5634 | Val: 0.5880

In [ ]:
pip install snntorch


In [ ]:
pip install spikingjelly


In [7]:
import sys
print(sys.version)        # vrei să vezi ce Python ai
import torch
print(torch.__version__)  # vrei PyTorch >= 2.2 pentru Python 3.12

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


ValueError: module functions cannot set METH_CLASS or METH_STATIC

In [10]:
"Model Hibrid de EfficientNet antrenat pe urban sound8k  cu un cap de SNN  si rate encoding"



import os
import copy
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import torchvision.models as models

from spikingjelly.activation_based import (
    neuron,
    functional,
    surrogate,
    layer
)

warnings.filterwarnings("ignore")


DATA_PATH  = "/kaggle/input/datasets/chrisfilo/urbansound8k"
CSV_PATH   = os.path.join(DATA_PATH, "UrbanSound8K.csv")
AUDIO_DIR  = DATA_PATH
MODELS_PATH = "/kaggle/input/datasets/biliutaadelina/trained-models"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 10
BATCH_SIZE  = 16
EPOCHS      = 20          # mai multe epoci; early stopping va opri dacă e cazul

T_SNN    = 6              # FIX: era 15; 4–6 e suficient și de 2.5× mai rapid
IMG_SIZE = 120

SAMPLE_RATE = 16000

# Transformări de bază (fără augmentare) — instanțiate o singură dată
MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_mels=128,
    n_fft=1024,
    hop_length=256
).to(DEVICE)

DB_TRANSFORM = torchaudio.transforms.AmplitudeToDB().to(DEVICE)

class AudioDataset(Dataset):

    def __init__(self, dataframe, audio_dir, apply_augment=False):
        self.df             = dataframe.reset_index(drop=True)
        self.audio_dir      = audio_dir
        self.apply_augment  = apply_augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        fold      = row["fold"]
        file_name = row["slice_file_name"]
        label     = row["classID"]
        path      = os.path.join(self.audio_dir, f"fold{fold}", file_name)

        waveform, sr = torchaudio.load(path)

        # mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # resample
        if sr != SAMPLE_RATE:
            waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)

        # fix la 4 secunde
        target_length = SAMPLE_RATE * 4
        if waveform.shape[1] < target_length:
            waveform = F.pad(waveform, (0, target_length - waveform.shape[1]))
        else:
            waveform = waveform[:, :target_length]

        # augmentare pe waveform (înainte de spectrogram)
        if self.apply_augment:
            waveform = waveform_augment(waveform)

        return waveform, label

def waveform_augment(wav):
    """Augmentări pe semnal brut: noise, gain, time shift."""

    # Gaussian noise ușor
    if random.random() < 0.5:
        noise = torch.randn_like(wav) * random.uniform(0.002, 0.008)
        wav   = wav + noise

    # Gain aleator ±6 dB
    if random.random() < 0.5:
        gain = random.uniform(0.5, 1.5)
        wav  = wav * gain

    # Time shift: deplasare circulară ±0.5 sec
    if random.random() < 0.5:
        shift = random.randint(-SAMPLE_RATE // 2, SAMPLE_RATE // 2)
        wav   = torch.roll(wav, shift, dims=-1)

    return wav
    
def wav_to_mel(wav, apply_augment=False):
    """Convertește waveform → mel spectrogram normalizat, gata pentru EfficientNet."""

    mel = MEL_TRANSFORM(wav)
    mel = DB_TRANSFORM(mel)

    if apply_augment:
        mel = spec_augment(mel)

    # Normalizare per-sample
    mean = mel.mean(dim=(-2, -1), keepdim=True)
    std  = mel.std(dim=(-2, -1), keepdim=True)
    mel  = (mel - mean) / (std + 1e-6)

    # Resize la IMG_SIZE × IMG_SIZE
    mel = F.interpolate(mel, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)

    # EfficientNet așteaptă 3 canale
    mel = mel.repeat(1, 3, 1, 1)

    return mel


def spec_augment(mel):
    # Time masking (2 măști independente)
    if random.random() < 0.8:
        mel = T.TimeMasking(time_mask_param=30)(mel)
    if random.random() < 0.5:
        mel = T.TimeMasking(time_mask_param=20)(mel)

    # Frequency masking (2 măști independente)
    if random.random() < 0.8:
        mel = T.FrequencyMasking(freq_mask_param=20)(mel)
    if random.random() < 0.5:
        mel = T.FrequencyMasking(freq_mask_param=15)(mel)

    return mel

def mixup_data(x, y, alpha=0.3):
    """Interpolează perechi de exemple; returnează mixed_x, y_a, y_b, lambda."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

class EarlyStopping:

    def __init__(self, patience=7, min_delta=1e-4):
        self.patience         = patience
        self.min_delta        = min_delta
        self.best_loss        = float("inf")
        self.counter          = 0
        self.best_model_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss        = val_loss
            self.counter          = 0
            self.best_model_state = copy.deepcopy(model.state_dict())
            return False
        self.counter += 1
        return self.counter >= self.patience

class HybridEfficientNetSNN(nn.Module):

    def __init__(self, checkpoint_path, num_classes=10, T=6, hidden_dim=512):
        super().__init__()

        self.T = T

        effnet = models.efficientnet_b2(weights=None)
        effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, num_classes)

        state_dict = torch.load(checkpoint_path, map_location="cpu")
        effnet.load_state_dict(state_dict)
        del state_dict
        torch.cuda.empty_cache()
        print(f"Loaded checkpoint: {checkpoint_path}")

        self.features  = effnet.features
        self.pool      = nn.AdaptiveAvgPool2d(1)
        feature_dim    = effnet.classifier[1].in_features  # 1408 pentru B2

        for param in self.features.parameters():
            param.requires_grad = False

        for param in self.features[-4:].parameters():
            param.requires_grad = True

        self.fc1     = layer.Linear(feature_dim, hidden_dim)
        # În loc de: self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn1 = nn.LayerNorm(hidden_dim)
        self.lif1    = neuron.LIFNode(
            tau=2.0,
            v_threshold=1.0,
            v_reset=0.0,
            surrogate_function=surrogate.ATan()
        )
        self.dropout = layer.Dropout(0.5)
        self.fc2     = layer.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # Backbone (grad oprit pentru blocurile înghețate)
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)  # (B, feature_dim)

        functional.reset_net(self)

        outputs = []
        for _ in range(self.T):
            h   = self.fc1(x)
            h   = self.bn1(h)
            h   = self.lif1(h)
            h   = self.dropout(h)
            out = self.fc2(h)
            outputs.append(out)

        # Medie peste timesteps
        logits = torch.stack(outputs).mean(0)
        return logits

def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x    = wav_to_mel(x, apply_augment=True)

        use_mixup = random.random() < 0.5
        if use_mixup:
            x, y_a, y_b, lam = mixup_data(x, y, alpha=0.3)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(x)
            if use_mixup:
                loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
            else:
                loss = criterion(logits, y)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == y).sum().item()
        total      += y.size(0)

    return total_loss / len(loader), correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y   = x.to(DEVICE), y.to(DEVICE)
        x      = wav_to_mel(x, apply_augment=False)
        logits = model(x)
        loss   = criterion(logits, y)

        total_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == y).sum().item()
        total      += y.size(0)

    return total_loss / len(loader), correct / total


@torch.no_grad()
def test_model(model, loader):
    model.eval()
    correct, total = 0, 0

    for x, y in loader:
        x, y   = x.to(DEVICE), y.to(DEVICE)
        x      = wav_to_mel(x, apply_augment=False)
        logits = model(x)
        preds  = torch.argmax(logits, dim=1)
        correct += (preds == y).sum().item()
        total   += y.size(0)

    return correct / total
    
def train_model(checkpoint_path, train_loader, val_loader):
    model = HybridEfficientNetSNN(
        checkpoint_path=checkpoint_path,
        num_classes=NUM_CLASSES,
        T=T_SNN
    ).to(DEVICE)
    
    backbone_params = list(model.features[-4:].parameters())
    head_params = (
        list(model.fc1.parameters()) +
        list(model.bn1.parameters()) +
        list(model.fc2.parameters())
    )

    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": 3e-5},
            {"params": head_params,     "lr": 1e-4},
        ],
        weight_decay=1e-4
    )

    warmup_epochs = 3

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(EPOCHS - warmup_epochs, 1)
        return 0.5 * (1.0 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # FIX: label_smoothing 0.1 → 0.15
    criterion    = nn.CrossEntropyLoss(label_smoothing=0.15)
    scaler       = torch.cuda.amp.GradScaler()
    # FIX: patience 5 → 7
    early_stopper = EarlyStopping(patience=7)
    best_val_acc  = 0.0

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler
        )
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        scheduler.step()

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        print(
            f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if early_stopper.step(val_loss, model):
            print("Early stopping triggered.")
            break

    model.load_state_dict(early_stopper.best_model_state)
    return model

def save_fold_model(model, fold, val_acc, test_acc, save_dir="/kaggle/working/saved_models"):
    """Salvează modelul și metadatele pentru un fold specific."""
    os.makedirs(save_dir, exist_ok=True)

    save_path = os.path.join(save_dir, f"hybrid_snn_fold_{fold}.pt")
    torch.save({
        "fold":       fold,
        "model_state_dict": model.state_dict(),
        "val_acc":    val_acc,
        "test_acc":   test_acc,
        "T_SNN":      T_SNN,
        "IMG_SIZE":   IMG_SIZE,
        "NUM_CLASSES": NUM_CLASSES,
    }, save_path)

    print(f"Model fold {fold} salvat → {save_path}")
    return save_path

def run_cross_validation():
    df             = pd.read_csv(CSV_PATH)
    all_accuracies = []

    for fold in range(1, 11):
        print("\n" + "=" * 70)
        print(f"FOLD {fold}")
        print("=" * 70)

        checkpoint_path = os.path.join(MODELS_PATH, f"model_fold_{fold}.pt")
        if not os.path.exists(checkpoint_path):
            print(f"Checkpoint missing: {checkpoint_path}")
            continue

        train_df_full = df[df["fold"] != fold]
        test_df       = df[df["fold"] == fold]

        train_df, val_df = train_test_split(
            train_df_full,
            test_size=0.1,
            stratify=train_df_full["classID"],
            random_state=42
        )

        # Dataset: augmentare waveform activată doar pentru train
        train_dataset = AudioDataset(train_df, AUDIO_DIR, apply_augment=True)
        val_dataset   = AudioDataset(val_df,   AUDIO_DIR, apply_augment=False)
        test_dataset  = AudioDataset(test_df,  AUDIO_DIR, apply_augment=False)

        train_loader = DataLoader(
            train_dataset, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=2, pin_memory=True
        )
        val_loader = DataLoader(
            val_dataset, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=2, pin_memory=True
        )
        test_loader = DataLoader(
            test_dataset, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=2, pin_memory=True
        )

        model = train_model(checkpoint_path, train_loader, val_loader)

        # Evaluare finală pe val și test
        _, final_val_acc = evaluate(model, val_loader, nn.CrossEntropyLoss(label_smoothing=0.15))
        test_acc = test_model(model, test_loader)

        print(f"\nFold {fold} Test Accuracy: {test_acc:.4f}")
        all_accuracies.append(test_acc)

        # Salvare checkpoint complet
        save_fold_model(model, fold, val_acc=final_val_acc, test_acc=test_acc)

        del model
        torch.cuda.empty_cache()

    # ── Rezultate finale ──────────────────────────────────────────────────────
    mean_acc = np.mean(all_accuracies)
    std_acc  = np.std(all_accuracies)

    print("\n" + "=" * 70)
    print("FINAL RESULTS")
    print("=" * 70)
    print(f"Per-fold accuracies : {[f'{a:.4f}' for a in all_accuracies]}")
    print(f"Mean Accuracy       : {mean_acc:.4f}")
    print(f"Std  Accuracy       : {std_acc:.4f}")

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.bar(range(1, len(all_accuracies) + 1), all_accuracies, color="steelblue")
    plt.axhline(mean_acc, color="red", linestyle="--", label=f"Mean {mean_acc:.4f}")
    plt.xlabel("Fold")
    plt.ylabel("Test Accuracy")
    plt.title("Per-fold accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.boxplot(all_accuracies)
    plt.ylabel("Accuracy")
    plt.title("Hybrid EfficientNet-B2 + SNN  (v2)")

    plt.tight_layout()
    plt.show()


# =============================================================================
if __name__ == "__main__":
    run_cross_validation()


FOLD 1
Loaded checkpoint: /kaggle/input/datasets/biliutaadelina/trained-models/model_fold_1.pt
Epoch [01/20] | Train Loss: 1.9876 | Train Acc: 0.4232 | Val Loss: 1.6749 | Val Acc: 0.7443
Epoch [02/20] | Train Loss: 1.5232 | Train Acc: 0.5674 | Val Loss: 1.2228 | Val Acc: 0.7964
Epoch [03/20] | Train Loss: 1.2930 | Train Acc: 0.6262 | Val Loss: 1.0186 | Val Acc: 0.8588
Epoch [04/20] | Train Loss: 1.2131 | Train Acc: 0.6801 | Val Loss: 0.9776 | Val Acc: 0.8791
Epoch [05/20] | Train Loss: 1.1878 | Train Acc: 0.6696 | Val Loss: 0.9849 | Val Acc: 0.8830
Epoch [06/20] | Train Loss: 1.1587 | Train Acc: 0.6775 | Val Loss: 0.9324 | Val Acc: 0.9008
Epoch [07/20] | Train Loss: 1.1176 | Train Acc: 0.6981 | Val Loss: 0.9152 | Val Acc: 0.9109
Epoch [08/20] | Train Loss: 1.1233 | Train Acc: 0.7109 | Val Loss: 0.8986 | Val Acc: 0.9109
Epoch [09/20] | Train Loss: 1.1189 | Train Acc: 0.6950 | Val Loss: 0.9056 | Val Acc: 0.9059
Epoch [10/20] | Train Loss: 1.1034 | Train Acc: 0.7092 | Val Loss: 0.9136 | 

KeyboardInterrupt: 

In [5]:

"Model Hibrid de EfficientNet antrenat pe urban sound8k  cu un cap de SNN  si latency encoding"


import os
import copy
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import torchvision.models as models

from spikingjelly.activation_based import (
    neuron,
    functional,
    surrogate,
    layer
)

warnings.filterwarnings("ignore")


DATA_PATH   = "/kaggle/input/datasets/chrisfilo/urbansound8k"
CSV_PATH    = os.path.join(DATA_PATH, "UrbanSound8K.csv")
AUDIO_DIR   = DATA_PATH
MODELS_PATH = "/kaggle/input/datasets/biliutaadelina/trained-models"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 10
BATCH_SIZE  = 16
EPOCHS      = 20

# Latency encoding are nevoie de mai multe timesteps pentru rezoluție temporală bună
T_SNN    = 10
IMG_SIZE = 120

SAMPLE_RATE = 16000

# Transformări de bază — instanțiate o singură dată
MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_mels=128,
    n_fft=1024,
    hop_length=256
).to(DEVICE)

DB_TRANSFORM = torchaudio.transforms.AmplitudeToDB().to(DEVICE)

class AudioDataset(Dataset):

    def __init__(self, dataframe, audio_dir, apply_augment=False):
        self.df            = dataframe.reset_index(drop=True)
        self.audio_dir     = audio_dir
        self.apply_augment = apply_augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        fold      = row["fold"]
        file_name = row["slice_file_name"]
        label     = row["classID"]
        path      = os.path.join(self.audio_dir, f"fold{fold}", file_name)

        waveform, sr = torchaudio.load(path)

        # mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # resample
        if sr != SAMPLE_RATE:
            waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)

        # fix la 4 secunde
        target_length = SAMPLE_RATE * 4
        if waveform.shape[1] < target_length:
            waveform = F.pad(waveform, (0, target_length - waveform.shape[1]))
        else:
            waveform = waveform[:, :target_length]

        if self.apply_augment:
            waveform = waveform_augment(waveform)

        return waveform, label

def waveform_augment(wav):
    """Augmentări pe semnal brut: noise, gain, time shift."""

    # Gaussian noise ușor
    if random.random() < 0.5:
        noise = torch.randn_like(wav) * random.uniform(0.002, 0.008)
        wav   = wav + noise

    # Gain aleator ±6 dB
    if random.random() < 0.5:
        gain = random.uniform(0.5, 1.5)
        wav  = wav * gain

    # Time shift: deplasare circulară ±0.5 sec
    if random.random() < 0.5:
        shift = random.randint(-SAMPLE_RATE // 2, SAMPLE_RATE // 2)
        wav   = torch.roll(wav, shift, dims=-1)

    return wav


def wav_to_mel(wav, apply_augment=False):
    """Convertește waveform → mel spectrogram normalizat, gata pentru EfficientNet."""

    mel = MEL_TRANSFORM(wav)
    mel = DB_TRANSFORM(mel)

    if apply_augment:
        mel = spec_augment(mel)

    # Normalizare per-sample
    mean = mel.mean(dim=(-2, -1), keepdim=True)
    std  = mel.std(dim=(-2, -1), keepdim=True)
    mel  = (mel - mean) / (std + 1e-6)

    # Resize la IMG_SIZE × IMG_SIZE
    mel = F.interpolate(mel, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)

    # EfficientNet așteaptă 3 canale
    mel = mel.repeat(1, 3, 1, 1)

    return mel


def spec_augment(mel):

    # Time masking (2 măști independente)
    if random.random() < 0.8:
        mel = T.TimeMasking(time_mask_param=30)(mel)
    if random.random() < 0.5:
        mel = T.TimeMasking(time_mask_param=20)(mel)

    # Frequency masking (2 măști independente)
    if random.random() < 0.8:
        mel = T.FrequencyMasking(freq_mask_param=20)(mel)
    if random.random() < 0.5:
        mel = T.FrequencyMasking(freq_mask_param=15)(mel)

    return mel

def latency_encode(x, T):
    # Normalizează fiecare sample în [0, 1] per feature_dim
    x_min  = x.min(dim=1, keepdim=True).values          # (B, 1)
    x_max  = x.max(dim=1, keepdim=True).values          # (B, 1)
    x_norm = (x - x_min) / (x_max - x_min + 1e-6)      # (B, feature_dim) ∈ [0,1]

    spike_times = torch.floor((1.0 - x_norm) * (T - 1)).long()  # (B, feature_dim)

    # Construiește spike train-ul binar
    spike_train = torch.zeros(T, x.size(0), x.size(1), device=x.device)
    for t in range(T):
        spike_train[t] = (spike_times == t).float()

    return spike_train  # (T, B, feature_dim)

def mixup_data(x, y, alpha=0.3):
    """Interpolează perechi de exemple; returnează mixed_x, y_a, y_b, lambda."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

class EarlyStopping:

    def __init__(self, patience=7, min_delta=1e-4):
        self.patience         = patience
        self.min_delta        = min_delta
        self.best_loss        = float("inf")
        self.counter          = 0
        self.best_model_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss        = val_loss
            self.counter          = 0
            self.best_model_state = copy.deepcopy(model.state_dict())
            return False
        self.counter += 1
        return self.counter >= self.patience

class HybridEfficientNetSNN(nn.Module):

    def __init__(self, checkpoint_path, num_classes=10, T=10, hidden_dim=512):
        super().__init__()

        self.T = T

        # ── Încarcă EfficientNet-B2 pre-antrenat (checkpoint propriu) ─────────
        effnet = models.efficientnet_b2(weights=None)
        effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, num_classes)

        state_dict = torch.load(checkpoint_path, map_location="cpu")
        effnet.load_state_dict(state_dict)
        del state_dict
        torch.cuda.empty_cache()
        print(f"Loaded checkpoint: {checkpoint_path}")

        self.features = effnet.features
        self.pool     = nn.AdaptiveAvgPool2d(1)
        feature_dim   = effnet.classifier[1].in_features  # 1408 pentru B2

        for param in self.features.parameters():
            param.requires_grad = False

        for param in self.features[-4:].parameters():
            param.requires_grad = True

        self.fc1  = layer.Linear(feature_dim, hidden_dim)
        self.bn1  = nn.LayerNorm(hidden_dim)
        self.lif1 = neuron.LIFNode(
            tau=1.5,         
            v_threshold=0.5,   
            v_reset=0.0,
            surrogate_function=surrogate.ATan()
        )
        self.dropout = layer.Dropout(0.5)
        self.fc2     = layer.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # ── Backbone: extrage features continue ───────────────────────────────
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)   # (B, feature_dim)

        functional.reset_net(self)

        spike_train = latency_encode(x, self.T)   # (T, B, feature_dim)

        outputs = []
        for t in range(self.T):
            h   = self.fc1(spike_train[t])   # input = spike-uri binare (0/1)
            h   = self.bn1(h)
            h   = self.lif1(h)
            h   = self.dropout(h)
            out = self.fc2(h)
            outputs.append(out)

        logits = torch.stack(outputs).sum(0)
        return logits

def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x    = wav_to_mel(x, apply_augment=True)

        use_mixup = random.random() < 0.5
        if use_mixup:
            x, y_a, y_b, lam = mixup_data(x, y, alpha=0.3)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(x)
            if use_mixup:
                loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
            else:
                loss = criterion(logits, y)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == y).sum().item()
        total      += y.size(0)

    return total_loss / len(loader), correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y   = x.to(DEVICE), y.to(DEVICE)
        x      = wav_to_mel(x, apply_augment=False)
        logits = model(x)
        loss   = criterion(logits, y)

        total_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == y).sum().item()
        total      += y.size(0)

    return total_loss / len(loader), correct / total


@torch.no_grad()
def test_model(model, loader):
    model.eval()
    correct, total = 0, 0

    for x, y in loader:
        x, y   = x.to(DEVICE), y.to(DEVICE)
        x      = wav_to_mel(x, apply_augment=False)
        logits = model(x)
        preds  = torch.argmax(logits, dim=1)
        correct += (preds == y).sum().item()
        total   += y.size(0)

    return correct / total

def train_model(checkpoint_path, train_loader, val_loader):

    model = HybridEfficientNetSNN(
        checkpoint_path=checkpoint_path,
        num_classes=NUM_CLASSES,
        T=T_SNN
    ).to(DEVICE)

    backbone_params = list(model.features[-4:].parameters())
    head_params = (
        list(model.fc1.parameters()) +
        list(model.bn1.parameters()) +
        list(model.fc2.parameters())
    )

    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": 3e-5},
            {"params": head_params,     "lr": 1e-4},
        ],
        weight_decay=1e-4
    )

    warmup_epochs = 3

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(EPOCHS - warmup_epochs, 1)
        return 0.5 * (1.0 + np.cos(np.pi * progress))

    scheduler     = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    criterion     = nn.CrossEntropyLoss(label_smoothing=0.15)
    scaler        = torch.cuda.amp.GradScaler()
    early_stopper = EarlyStopping(patience=7)
    best_val_acc  = 0.0

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler
        )
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        scheduler.step()

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        print(
            f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if early_stopper.step(val_loss, model):
            print("Early stopping triggered.")
            break

    model.load_state_dict(early_stopper.best_model_state)
    return model

def run_cross_validation():
    df             = pd.read_csv(CSV_PATH)
    all_accuracies = []

    for fold in range(1, 11):
        print("\n" + "=" * 70)
        print(f"FOLD {fold}")
        print("=" * 70)

        checkpoint_path = os.path.join(MODELS_PATH, f"model_fold_{fold}.pt")
        if not os.path.exists(checkpoint_path):
            print(f"Checkpoint missing: {checkpoint_path}")
            continue

        train_df_full = df[df["fold"] != fold]
        test_df       = df[df["fold"] == fold]

        train_df, val_df = train_test_split(
            train_df_full,
            test_size=0.1,
            stratify=train_df_full["classID"],
            random_state=42
        )

        train_dataset = AudioDataset(train_df, AUDIO_DIR, apply_augment=True)
        val_dataset   = AudioDataset(val_df,   AUDIO_DIR, apply_augment=False)
        test_dataset  = AudioDataset(test_df,  AUDIO_DIR, apply_augment=False)

        train_loader = DataLoader(
            train_dataset, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=2, pin_memory=True
        )
        val_loader = DataLoader(
            val_dataset, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=2, pin_memory=True
        )
        test_loader = DataLoader(
            test_dataset, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=2, pin_memory=True
        )

        model    = train_model(checkpoint_path, train_loader, val_loader)
        test_acc = test_model(model, test_loader)

        print(f"\nFold {fold} Test Accuracy: {test_acc:.4f}")
        all_accuracies.append(test_acc)

        del model
        torch.cuda.empty_cache()

    mean_acc = np.mean(all_accuracies)
    std_acc  = np.std(all_accuracies)

    print("\n" + "=" * 70)
    print("FINAL RESULTS")
    print("=" * 70)
    print(f"Per-fold accuracies : {[f'{a:.4f}' for a in all_accuracies]}")
    print(f"Mean Accuracy       : {mean_acc:.4f}")
    print(f"Std  Accuracy       : {std_acc:.4f}")

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.bar(range(1, len(all_accuracies) + 1), all_accuracies, color="steelblue")
    plt.axhline(mean_acc, color="red", linestyle="--", label=f"Mean {mean_acc:.4f}")
    plt.xlabel("Fold")
    plt.ylabel("Test Accuracy")
    plt.title("Per-fold accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.boxplot(all_accuracies)
    plt.ylabel("Accuracy")
    plt.title("Hybrid EfficientNet-B2 + SNN  (Latency Encoding)")

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    run_cross_validation()


FOLD 1
Loaded checkpoint: /kaggle/input/datasets/biliutaadelina/trained-models/model_fold_1.pt
Epoch [01/20] | Train Loss: 2.5383 | Train Acc: 0.2584 | Val Loss: 1.4095 | Val Acc: 0.6781
Epoch [02/20] | Train Loss: 1.9431 | Train Acc: 0.4486 | Val Loss: 1.3388 | Val Acc: 0.7061
Epoch [03/20] | Train Loss: 1.7291 | Train Acc: 0.4926 | Val Loss: 1.2414 | Val Acc: 0.7646
Epoch [04/20] | Train Loss: 1.5817 | Train Acc: 0.5220 | Val Loss: 1.2152 | Val Acc: 0.7684
Epoch [05/20] | Train Loss: 1.5390 | Train Acc: 0.5258 | Val Loss: 1.2046 | Val Acc: 0.7748
Epoch [06/20] | Train Loss: 1.4923 | Train Acc: 0.5583 | Val Loss: 1.2556 | Val Acc: 0.7532
Epoch [07/20] | Train Loss: 1.4480 | Train Acc: 0.5652 | Val Loss: 1.2091 | Val Acc: 0.7684
Epoch [08/20] | Train Loss: 1.4364 | Train Acc: 0.5679 | Val Loss: 1.1749 | Val Acc: 0.7888
Epoch [09/20] | Train Loss: 1.4263 | Train Acc: 0.5592 | Val Loss: 1.1595 | Val Acc: 0.8130
Epoch [10/20] | Train Loss: 1.3857 | Train Acc: 0.5931 | Val Loss: 1.1779 | 

KeyboardInterrupt: 

In [ ]:
import os
import copy
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import torchvision.models as models

from spikingjelly.activation_based import (
    neuron,
    functional,
    surrogate,
    layer
)

warnings.filterwarnings("ignore")


DATA_PATH   = "/kaggle/input/datasets/chrisfilo/urbansound8k"
CSV_PATH    = os.path.join(DATA_PATH, "UrbanSound8K.csv")
AUDIO_DIR   = DATA_PATH
MODELS_PATH = "/kaggle/input/datasets/biliutaadelina/trained-models"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 10
BATCH_SIZE  = 16
EPOCHS      = 20

# Latency encoding are nevoie de mai multe timesteps pentru rezoluție temporală bună
T_SNN    = 10
IMG_SIZE = 120

SAMPLE_RATE = 16000

# Transformări de bază — instanțiate o singură dată
MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_mels=128,
    n_fft=1024,
    hop_length=256
).to(DEVICE)

DB_TRANSFORM = torchaudio.transforms.AmplitudeToDB().to(DEVICE)

# =============================================================================
# AUDIO DATASET
# =============================================================================

class AudioDataset(Dataset):

    def __init__(self, dataframe, audio_dir, apply_augment=False):
        self.df            = dataframe.reset_index(drop=True)
        self.audio_dir     = audio_dir
        self.apply_augment = apply_augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        fold      = row["fold"]
        file_name = row["slice_file_name"]
        label     = row["classID"]
        path      = os.path.join(self.audio_dir, f"fold{fold}", file_name)

        waveform, sr = torchaudio.load(path)

        # mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # resample
        if sr != SAMPLE_RATE:
            waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)

        # fix la 4 secunde
        target_length = SAMPLE_RATE * 4
        if waveform.shape[1] < target_length:
            waveform = F.pad(waveform, (0, target_length - waveform.shape[1]))
        else:
            waveform = waveform[:, :target_length]

        # augmentare pe waveform (înainte de spectrogram)
        if self.apply_augment:
            waveform = waveform_augment(waveform)

        return waveform, label

# =============================================================================
# AUGMENTARE WAVEFORM  (rulează pe CPU, în Dataset)
# =============================================================================

def waveform_augment(wav):
    """Augmentări pe semnal brut: noise, gain, time shift."""

    # Gaussian noise ușor
    if random.random() < 0.5:
        noise = torch.randn_like(wav) * random.uniform(0.002, 0.008)
        wav   = wav + noise

    # Gain aleator ±6 dB
    if random.random() < 0.5:
        gain = random.uniform(0.5, 1.5)
        wav  = wav * gain

    # Time shift: deplasare circulară ±0.5 sec
    if random.random() < 0.5:
        shift = random.randint(-SAMPLE_RATE // 2, SAMPLE_RATE // 2)
        wav   = torch.roll(wav, shift, dims=-1)

    return wav

# =============================================================================
# SPECTROGRAM + SPEC-AUGMENT  (rulează pe GPU, în training loop)
# =============================================================================

def wav_to_mel(wav, apply_augment=False):
    """Convertește waveform → mel spectrogram normalizat, gata pentru EfficientNet."""

    mel = MEL_TRANSFORM(wav)
    mel = DB_TRANSFORM(mel)

    if apply_augment:
        mel = spec_augment(mel)

    # Normalizare per-sample
    mean = mel.mean(dim=(-2, -1), keepdim=True)
    std  = mel.std(dim=(-2, -1), keepdim=True)
    mel  = (mel - mean) / (std + 1e-6)

    # Resize la IMG_SIZE × IMG_SIZE
    mel = F.interpolate(mel, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)

    # EfficientNet așteaptă 3 canale
    mel = mel.repeat(1, 3, 1, 1)

    return mel


def spec_augment(mel):
    """SpecAugment: time masking + frequency masking, aplicate de 2 ori fiecare."""

    # Time masking (2 măști independente)
    if random.random() < 0.8:
        mel = T.TimeMasking(time_mask_param=30)(mel)
    if random.random() < 0.5:
        mel = T.TimeMasking(time_mask_param=20)(mel)

    # Frequency masking (2 măști independente)
    if random.random() < 0.8:
        mel = T.FrequencyMasking(freq_mask_param=20)(mel)
    if random.random() < 0.5:
        mel = T.FrequencyMasking(freq_mask_param=15)(mel)

    return mel

# =============================================================================
# LATENCY ENCODING
# =============================================================================

def latency_encode(x, T):
    """
    Convertește features continue în spike trains cu latency encoding.

    Principiu:
      - Fiecare neuron emite exact UN singur spike pe parcursul celor T timesteps.
      - Stimul puternic (valoare mare) → spike devreme (t=0).
      - Stimul slab    (valoare mică)  → spike târziu  (t=T-1).

    Args:
        x : (B, feature_dim) — features continue din backbone, după flatten
        T : numărul de timesteps

    Returns:
        spike_train : (T, B, feature_dim) — tensor binar (0/1),
                      exact un 1 per neuron per sample de-a lungul axei T
    """
    # Normalizează fiecare sample în [0, 1] per feature_dim
    x_min  = x.min(dim=1, keepdim=True).values          # (B, 1)
    x_max  = x.max(dim=1, keepdim=True).values          # (B, 1)
    x_norm = (x - x_min) / (x_max - x_min + 1e-6)      # (B, feature_dim) ∈ [0,1]

    # Calculează timestep-ul de spike pentru fiecare neuron
    # x_norm=1 (puternic) → spike_time=0   (primul timestep)
    # x_norm=0 (slab)     → spike_time=T-1 (ultimul timestep)
    spike_times = torch.floor((1.0 - x_norm) * (T - 1)).long()  # (B, feature_dim)

    # Construiește spike train-ul binar
    spike_train = torch.zeros(T, x.size(0), x.size(1), device=x.device)
    for t in range(T):
        spike_train[t] = (spike_times == t).float()

    return spike_train  # (T, B, feature_dim)

# =============================================================================
# MIXUP
# =============================================================================

def mixup_data(x, y, alpha=0.3):
    """Interpolează perechi de exemple; returnează mixed_x, y_a, y_b, lambda."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# =============================================================================
# EARLY STOPPING
# =============================================================================

class EarlyStopping:

    def __init__(self, patience=7, min_delta=1e-4):
        self.patience         = patience
        self.min_delta        = min_delta
        self.best_loss        = float("inf")
        self.counter          = 0
        self.best_model_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss        = val_loss
            self.counter          = 0
            self.best_model_state = copy.deepcopy(model.state_dict())
            return False
        self.counter += 1
        return self.counter >= self.patience

# =============================================================================
# MODEL: Hybrid EfficientNet-B2 + SNN  (Latency Encoding)
# =============================================================================

class HybridEfficientNetSNN(nn.Module):

    def __init__(self, checkpoint_path, num_classes=10, T=10, hidden_dim=512):
        super().__init__()

        self.T = T

        # ── Încarcă EfficientNet-B2 pre-antrenat (checkpoint propriu) ─────────
        effnet = models.efficientnet_b2(weights=None)
        effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, num_classes)

        state_dict = torch.load(checkpoint_path, map_location="cpu")
        effnet.load_state_dict(state_dict)
        del state_dict
        torch.cuda.empty_cache()
        print(f"Loaded checkpoint: {checkpoint_path}")

        self.features = effnet.features
        self.pool     = nn.AdaptiveAvgPool2d(1)
        feature_dim   = effnet.classifier[1].in_features  # 1408 pentru B2

        # ── Îngheață tot backbone-ul inițial ──────────────────────────────────
        for param in self.features.parameters():
            param.requires_grad = False

        # Dezgheță ultimele 4 blocuri pentru fine-tuning
        for param in self.features[-4:].parameters():
            param.requires_grad = True

        # ── Capul SNN ─────────────────────────────────────────────────────────
        self.fc1  = layer.Linear(feature_dim, hidden_dim)
        self.bn1  = nn.LayerNorm(hidden_dim)
        self.lif1 = neuron.LIFNode(
            tau=1.5,           # memorie mai scurtă față de rate encoding (era 2.0)
                               # → fiecare spike este procesat relativ independent
            v_threshold=0.5,   # prag mai mic față de rate encoding (era 1.0)
                               # → sensibilitate mai mare la spike-urile rare
            v_reset=0.0,
            surrogate_function=surrogate.ATan()
        )
        self.dropout = layer.Dropout(0.5)
        self.fc2     = layer.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # ── Backbone: extrage features continue ───────────────────────────────
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)   # (B, feature_dim)

        functional.reset_net(self)

        # ── Latency encoding: features continue → spike train binar ──────────
        # spike_train[t] conține 1 pentru neuronii care emit la timestep-ul t,
        # 0 pentru toți ceilalți. Fiecare neuron emite exact un singur spike.
        spike_train = latency_encode(x, self.T)   # (T, B, feature_dim)

        # ── Procesare SNN: fiecare timestep primește spike-urile corespunzătoare
        outputs = []
        for t in range(self.T):
            h   = self.fc1(spike_train[t])   # input = spike-uri binare (0/1)
            h   = self.bn1(h)
            h   = self.lif1(h)
            h   = self.dropout(h)
            out = self.fc2(h)
            outputs.append(out)

        # ── Agregare: SUMA în loc de medie ────────────────────────────────────
        # În latency encoding suma este preferată față de medie deoarece:
        #   - spike-urile timpurii (stimuli puternici) contribuie la mai mulți
        #     timesteps prin dinamica LIF (potențialul se acumulează)
        #   - suma păstrează magnitudinea totală a activării
        logits = torch.stack(outputs).sum(0)
        return logits

def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x    = wav_to_mel(x, apply_augment=True)

        use_mixup = random.random() < 0.5
        if use_mixup:
            x, y_a, y_b, lam = mixup_data(x, y, alpha=0.3)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(x)
            if use_mixup:
                loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
            else:
                loss = criterion(logits, y)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == y).sum().item()
        total      += y.size(0)

    return total_loss / len(loader), correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y   = x.to(DEVICE), y.to(DEVICE)
        x      = wav_to_mel(x, apply_augment=False)
        logits = model(x)
        loss   = criterion(logits, y)

        total_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == y).sum().item()
        total      += y.size(0)

    return total_loss / len(loader), correct / total


@torch.no_grad()
def test_model(model, loader):
    model.eval()
    correct, total = 0, 0

    for x, y in loader:
        x, y   = x.to(DEVICE), y.to(DEVICE)
        x      = wav_to_mel(x, apply_augment=False)
        logits = model(x)
        preds  = torch.argmax(logits, dim=1)
        correct += (preds == y).sum().item()
        total   += y.size(0)

    return correct / total

def train_model(checkpoint_path, train_loader, val_loader):

    model = HybridEfficientNetSNN(
        checkpoint_path=checkpoint_path,
        num_classes=NUM_CLASSES,
        T=T_SNN
    ).to(DEVICE)

    # Discriminative learning rates:
    #   backbone dezghețat → LR mic  (3e-5)
    #   capul SNN          → LR mare (1e-4)
    backbone_params = list(model.features[-4:].parameters())
    head_params = (
        list(model.fc1.parameters()) +
        list(model.bn1.parameters()) +
        list(model.fc2.parameters())
    )

    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": 3e-5},
            {"params": head_params,     "lr": 1e-4},
        ],
        weight_decay=1e-4
    )

    # Warmup 3 epoci + cosine decay
    warmup_epochs = 3

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(EPOCHS - warmup_epochs, 1)
        return 0.5 * (1.0 + np.cos(np.pi * progress))

    scheduler     = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    criterion     = nn.CrossEntropyLoss(label_smoothing=0.15)
    scaler        = torch.cuda.amp.GradScaler()
    early_stopper = EarlyStopping(patience=7)
    best_val_acc  = 0.0

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler
        )
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        scheduler.step()

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        print(
            f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if early_stopper.step(val_loss, model):
            print("Early stopping triggered.")
            break

    model.load_state_dict(early_stopper.best_model_state)
    return model

# =============================================================================
# CROSS-VALIDATION  (10-fold oficial UrbanSound8K)
# =============================================================================

def run_cross_validation():
    df             = pd.read_csv(CSV_PATH)
    all_accuracies = []

    for fold in range(1, 11):
        print("\n" + "=" * 70)
        print(f"FOLD {fold}")
        print("=" * 70)

        checkpoint_path = os.path.join(MODELS_PATH, f"model_fold_{fold}.pt")
        if not os.path.exists(checkpoint_path):
            print(f"Checkpoint missing: {checkpoint_path}")
            continue

        train_df_full = df[df["fold"] != fold]
        test_df       = df[df["fold"] == fold]

        train_df, val_df = train_test_split(
            train_df_full,
            test_size=0.1,
            stratify=train_df_full["classID"],
            random_state=42
        )

        train_dataset = AudioDataset(train_df, AUDIO_DIR, apply_augment=True)
        val_dataset   = AudioDataset(val_df,   AUDIO_DIR, apply_augment=False)
        test_dataset  = AudioDataset(test_df,  AUDIO_DIR, apply_augment=False)

        train_loader = DataLoader(
            train_dataset, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=2, pin_memory=True
        )
        val_loader = DataLoader(
            val_dataset, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=2, pin_memory=True
        )
        test_loader = DataLoader(
            test_dataset, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=2, pin_memory=True
        )

        model    = train_model(checkpoint_path, train_loader, val_loader)
        test_acc = test_model(model, test_loader)

        print(f"\nFold {fold} Test Accuracy: {test_acc:.4f}")
        all_accuracies.append(test_acc)

        del model
        torch.cuda.empty_cache()

    # ── Rezultate finale ──────────────────────────────────────────────────────
    mean_acc = np.mean(all_accuracies)
    std_acc  = np.std(all_accuracies)

    print("\n" + "=" * 70)
    print("FINAL RESULTS")
    print("=" * 70)
    print(f"Per-fold accuracies : {[f'{a:.4f}' for a in all_accuracies]}")
    print(f"Mean Accuracy       : {mean_acc:.4f}")
    print(f"Std  Accuracy       : {std_acc:.4f}")

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.bar(range(1, len(all_accuracies) + 1), all_accuracies, color="steelblue")
    plt.axhline(mean_acc, color="red", linestyle="--", label=f"Mean {mean_acc:.4f}")
    plt.xlabel("Fold")
    plt.ylabel("Test Accuracy")
    plt.title("Per-fold accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.boxplot(all_accuracies)
    plt.ylabel("Accuracy")
    plt.title("Hybrid EfficientNet-B2 + SNN  (Latency Encoding)")

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    run_cross_validation()

In [11]:
"""
SNN (Spiking Neural Network) cu latency si rate encoding - echivalent cu CNNClassifier
Foloseste latency encoding cu T=15 timesteps si SpikingJelly.

Arhitectura:
  - Latency Encoding: valori mai mari din mel spectrogram → spike mai devreme
  - 4 blocuri reziduale cu neuroni LIF (Leaky Integrate-and-Fire)
  - Output: suma spike-urilor pe timesteps → CrossEntropyLoss
  - Antrenare: BPTT (Backpropagation Through Time) cu surrogate gradient
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time


from spikingjelly.activation_based import neuron, functional, surrogate, layer

T = 15           # numar de timesteps (latency encoding)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def rate_encode(x: torch.Tensor, T: int = 15) -> torch.Tensor:
    x = torch.clamp(x, -3.0, 3.0)
    x_min = x.flatten(1).min(dim=1)[0].view(-1, 1, 1, 1)
    x_max = x.flatten(1).max(dim=1)[0].view(-1, 1, 1, 1)
    # Normalizare în [0, 1] — aceasta devine probabilitatea de fire
    x_norm = (x - x_min) / (x_max - x_min + 1e-6)

    # Generează T mostre Bernoulli independente pentru fiecare pixel
    # x_norm: (B, C, H, W) → expand la (T, B, C, H, W)
    x_expanded = x_norm.unsqueeze(0).expand(T, -1, -1, -1, -1)
    spike_train = torch.bernoulli(x_expanded)
    return spike_train


def latency_encode(x: torch.Tensor, T: int = 15) -> torch.Tensor:
    # Normalizeaza per sample (deja faci asta in dataset), dar
    # clipeaza valorile extreme inainte de encoding
    x = torch.clamp(x, -3.0, 3.0)  # mel-urile sunt deja normalizate z-score
    x_min = x.flatten(1).min(dim=1)[0].view(-1, 1, 1, 1)
    x_max = x.flatten(1).max(dim=1)[0].view(-1, 1, 1, 1)
    x_norm = (x - x_min) / (x_max - x_min + 1e-6)

    t_spike = torch.round((1.0 - x_norm) * (T - 1)).long()
    t_indices = torch.arange(T, device=x.device).view(T, 1, 1, 1, 1)
    spike_train = (t_indices == t_spike.unsqueeze(0)).float()
    return spike_train

class SNNResidualBlock(nn.Module):

    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, dropout: float = 0.1):
        super().__init__()

        sg = surrogate.ATan()

        self.bn1   = layer.BatchNorm2d(in_channels)
        self.conv1 = layer.Conv2d(in_channels, out_channels, kernel_size=3,
                                   stride=stride, padding=1, bias=False)
        self.lif1  = neuron.LIFNode(surrogate_function=sg, detach_reset=True)

        self.bn2      = layer.BatchNorm2d(out_channels)
        self.conv2    = layer.Conv2d(out_channels, out_channels, kernel_size=3,
                                      padding=1, bias=False)
        self.dropout  = layer.Dropout2d(p=dropout)
        self.lif2     = neuron.LIFNode(surrogate_function=sg, detach_reset=True)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                layer.Conv2d(in_channels, out_channels, kernel_size=1,
                              stride=stride, bias=False),
                layer.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.bn1(x)
        out = self.conv1(out)
        out = self.lif1(out)

        out = self.bn2(out)
        out = self.conv2(out)
        out = self.dropout(out)

        out = out + identity
        out = self.lif2(out)

        return out

class SNNClassifier(nn.Module):

    def __init__(self, num_classes: int = 10, T: int = 15):
        super().__init__()
        self.T = T

        self.block1 = SNNResidualBlock(1,   32,  stride=2, dropout=0.1)
        self.block2 = SNNResidualBlock(32,  64,  stride=2, dropout=0.2)
        self.block3 = SNNResidualBlock(64,  128, stride=2, dropout=0.3)
        self.block4 = SNNResidualBlock(128, 256, stride=2, dropout=0.3)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

        functional.set_step_mode(self, step_mode='m')

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        functional.reset_net(self)

        spike_input = latency_encode(x, T=self.T)
        #spike_input = rate_encode(x, T=self.T)   

        out = self.block1(spike_input)
        out = self.block2(out)
        out = self.block3(out)
        out = self.block4(out)

        T_curr, B, C, H, W = out.shape
        out = out.view(T_curr * B, C, H, W)
        out = self.pool(out)
        out = out.view(T_curr, B, C)

        out = out.sum(0)

        return self.classifier(out)



import os
import random
import copy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torchaudio
import torchaudio.transforms as T_audio
import torchaudio.transforms as T_aug
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

DATA_PATH  = "/kaggle/input/datasets/chrisfilo/urbansound8k"
CSV_PATH   = os.path.join(DATA_PATH, "UrbanSound8K.csv")
AUDIO_DIR  = DATA_PATH
BATCH_SIZE = 32
EPOCHS     = 30
LR         = 1e-3
T_STEPS    = 15

MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_mels=124,
    n_fft=1024,
    hop_length=512,
).to(DEVICE)

DB_TRANSFORM = torchaudio.transforms.AmplitudeToDB().to(DEVICE)
TIME_MASK    = T_aug.TimeMasking(time_mask_param=20)
FREQ_MASK    = T_aug.FrequencyMasking(freq_mask_param=15)


def apply_spec_augment(mel):
    if random.random() < 0.8:
        mel = TIME_MASK(mel)
    if random.random() < 0.8:
        mel = FREQ_MASK(mel)
    return mel


def wav_to_mel(wav):
    """Calculeaza mel spectrogram fara augmentare — folosit doar la pre-calcul."""
    mel = MEL_TRANSFORM(wav)
    mel = DB_TRANSFORM(mel)
    return mel


class CachedAudioDataset(Dataset):
    """
    Încarcă toate fișierele audio o singură dată în RAM ca mel spectrograme.
    __getitem__ face doar acces RAM + augmentare opțională — zero disk I/O.
    """

    def __init__(self, df, audio_dir, target_sr=16000, duration=4.0, apply_augment=False):
        self.df = df.reset_index(drop=True)
        self.apply_augment = apply_augment
        self.target_sr = target_sr
        self.target_len = int(target_sr * duration)

        # ── Încarcă TOT în RAM la inițializare ──
        print(f"  Încarc {len(self.df)} fișiere audio în RAM...")
        t0 = time.time()
        self.mels = []
        self.labels = []

        for idx in range(len(self.df)):
            row = self.df.iloc[idx]
            fold_dir = f"fold{row['fold']}"
            path = os.path.join(audio_dir, fold_dir, row['slice_file_name'])

            wav, sr = torchaudio.load(path)

            # Mono
            if wav.shape[0] > 1:
                wav = wav.mean(dim=0, keepdim=True)

            # Resample
            if sr != self.target_sr:
                wav = torchaudio.functional.resample(wav, sr, self.target_sr)

            # Padding / crop
            if wav.shape[1] < self.target_len:
                wav = F.pad(wav, (0, self.target_len - wav.shape[1]))
            else:
                wav = wav[:, :self.target_len]

            # Mel spectrogram fara augmentare — augmentarea se aplica in __getitem__
            mel = wav_to_mel(wav.to(DEVICE)).cpu()

            self.mels.append(mel)
            self.labels.append(int(row['classID']))

        print(f"  Gata — {len(self.mels)} mel spectrograme în RAM ({time.time()-t0:.1f}s)")

    def __len__(self):
        return len(self.mels)

    def __getitem__(self, idx):
        mel = self.mels[idx]    # acces RAM, zero disk I/O

        # Augmentare on-the-fly (diferita la fiecare epoca)
        if self.apply_augment:
            mel = apply_spec_augment(mel)

        # Normalizare per sample
        mean = mel.mean(dim=(-2, -1), keepdim=True)
        std  = mel.std(dim=(-2, -1), keepdim=True)
        mel  = (mel - mean) / (std + 1e-6)

        return mel, self.labels[idx]


# ── Early Stopping ───────────────────────────

class EarlyStopping:
    def __init__(self, patience=4, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_loss  = float("inf")
        self.counter    = 0
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.counter    = 0
            actual_model = model.module if isinstance(model, nn.DataParallel) else model
            self.best_state = copy.deepcopy(actual_model.state_dict())
            return False
        self.counter += 1
        return self.counter >= self.patience

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for mel, y in loader:
        mel, y = mel.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(mel)
        loss   = criterion(logits, y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def validate(model, loader, criterion):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for mel, y in loader:
            mel, y = mel.to(DEVICE), y.to(DEVICE)
            logits = model(mel)
            loss   = criterion(logits, y)
            total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    correct = 0
    total   = 0

    with torch.no_grad():
        for mel, y in loader:
            mel, y = mel.to(DEVICE), y.to(DEVICE)
            logits = model(mel)
            preds  = torch.argmax(logits, dim=1)
            correct += (preds == y).sum().item()
            total   += y.size(0)

    return correct / total

def run_cross_validation():
    df = pd.read_csv(CSV_PATH)
    all_accuracies = []

    for fold in range(1, 11):
        print(f"\n========== FOLD {fold} ==========")

        train_df_full = df[df["fold"] != fold]
        test_df       = df[df["fold"] == fold]

        train_df, val_df = train_test_split(
            train_df_full,
            test_size=0.1,
            stratify=train_df_full["classID"],
            random_state=42
        )

        # CachedAudioDataset încarcă totul în RAM — zero disk I/O în bucla de antrenare
        train_dataset = CachedAudioDataset(train_df, AUDIO_DIR, apply_augment=True)
        val_dataset   = CachedAudioDataset(val_df,   AUDIO_DIR, apply_augment=False)
        test_dataset  = CachedAudioDataset(test_df,  AUDIO_DIR, apply_augment=False)

        # num_workers=0 — datele sunt deja în RAM, nu mai avem nevoie de procese extra
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=0, pin_memory=True)
        val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=0, pin_memory=True)
        test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=0, pin_memory=True)

        model = SNNClassifier(num_classes=10, T=T_STEPS)

        if torch.cuda.device_count() > 1:
            print(f"-> Rulam in paralel pe {torch.cuda.device_count()} GPU-uri!")
            model = nn.DataParallel(model)

        model = model.to(DEVICE)

        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=3e-4)   # era 3e-4

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=6
        )

        criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

        early_stopper = EarlyStopping(patience=7)

        for epoch in range(EPOCHS):
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
            val_loss   = validate(model, val_loader, criterion)
            scheduler.step(val_loss)

            print(f"Epoch {epoch+1}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

            if early_stopper.step(val_loss, model):
                print("Early stopping triggered")
                break

        actual_model = model.module if isinstance(model, nn.DataParallel) else model
        actual_model.load_state_dict(early_stopper.best_state)

        acc = evaluate(model, test_loader)
        print(f"Fold {fold} Accuracy: {acc:.4f}")
        all_accuracies.append(acc)

    mean_acc = np.mean(all_accuracies)
    std_acc  = np.std(all_accuracies)

    print("\n========== FINAL RESULTS ==========")
    print(f"Mean Accuracy: {mean_acc:.4f}")
    print(f"Std Dev:       {std_acc:.4f}")

    plt.boxplot(all_accuracies)
    plt.title("10-Fold Cross Validation Accuracy (SNN)")
    plt.ylabel("Accuracy")
    plt.show()


if __name__ == "__main__":
    run_cross_validation()



========== FOLD 1 ==========
  Încarc 7073 fișiere audio în RAM...
  Gata — 7073 mel spectrograme în RAM (100.7s)
  Încarc 786 fișiere audio în RAM...
  Gata — 786 mel spectrograme în RAM (11.1s)
  Încarc 873 fișiere audio în RAM...
  Gata — 873 mel spectrograme în RAM (12.3s)
-> Rulam in paralel pe 2 GPU-uri!
Epoch 1/30 | Train: 2.1843 | Val: 1.9701
Epoch 2/30 | Train: 1.9995 | Val: 2.4515
Epoch 3/30 | Train: 1.9048 | Val: 2.0135
Epoch 4/30 | Train: 1.8410 | Val: 1.6383
Epoch 5/30 | Train: 1.8031 | Val: 1.7059
Epoch 6/30 | Train: 1.7663 | Val: 1.9027
Epoch 7/30 | Train: 1.7298 | Val: 1.5284
Epoch 8/30 | Train: 1.6944 | Val: 1.7277
Epoch 9/30 | Train: 1.6773 | Val: 1.4974
Epoch 10/30 | Train: 1.6204 | Val: 1.3579
Epoch 11/30 | Train: 1.5988 | Val: 1.3407
Epoch 12/30 | Train: 1.5542 | Val: 1.5067
Epoch 13/30 | Train: 1.4956 | Val: 1.2891
Epoch 14/30 | Train: 1.4636 | Val: 1.2381
Epoch 15/30 | Train: 1.4116 | Val: 1.2228
Epoch 16/30 | Train: 1.3861 | Val: 1.1907
Epoch 17/30 | Train: 1.3

KeyboardInterrupt: 

In [ ]:


"""
SNN Fully-Connected (fără convoluții)  cu latency encoding pentru UrbanSound8K
==========================================================
Arhitectura:
  - Mel spectrogram redus: 64 mels × 32 frames = 2048 features (aplatizat)
  - Latency encoding: T=20 timesteps
  - 3 blocuri FC-LIF cu skip connections (shortcut proiectat)
  - Suma spike-urilor pe timesteps → CrossEntropyLoss + label smoothing
Instalare:
    pip install spikingjelly torch torchaudio scikit-learn pandas matplotlib
"""

import os
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T_aug

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from spikingjelly.activation_based import neuron, functional, surrogate, layer

DATA_PATH  = "/kaggle/input/datasets/chrisfilo/urbansound8k"
CSV_PATH   = os.path.join(DATA_PATH, "UrbanSound8K.csv")
AUDIO_DIR  = DATA_PATH

T_STEPS    = 32       # timesteps pentru latency encoding
N_MELS     = 64       # mel bins (redus față de 128 din CNN — suficient pentru FC)
N_FRAMES   = 32       # număr de frames după hop_length=512, sr=16000, dur=4s → ~125 frames
                       # vom interpola/crop la 32 pentru a controla dimensionalitatea
INPUT_DIM  = N_MELS * N_FRAMES   # 64 × 32 = 2048

BATCH_SIZE = 32       # FC e mai eficient — putem mări batch-ul
EPOCHS     = 40
LR         = 1e-3
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")
print(f"Input dimensiune: {N_MELS}×{N_FRAMES} = {INPUT_DIM}")


def latency_encode(x: torch.Tensor, T: int = 15) -> torch.Tensor:

    x_min = x.min(dim=1, keepdim=True)[0]
    x_max = x.max(dim=1, keepdim=True)[0]
    x_norm = (x - x_min) / (x_max - x_min + 1e-6)  # (B, D) în [0, 1]

    # t_spike ∈ {0, ..., T-1}: stimul maxim → spike imediat (t=0)
    t_spike = torch.round((1.0 - x_norm) * (T - 1)).long()  # (B, D)

    # Construiește spike train
    t_idx = torch.arange(T, device=x.device).view(T, 1, 1)  # (T, 1, 1)
    t_spike_exp = t_spike.unsqueeze(0)                        # (1, B, D)
    spike_train = (t_idx == t_spike_exp).float()              # (T, B, D)

    return spike_train

class FCAlphaBlock(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.3):
        super().__init__()
        sg = surrogate.ATan(alpha=2.0)

        self.bn1  = nn.BatchNorm1d(in_dim)
        self.fc1  = layer.Linear(in_dim, out_dim, bias=False)
        # Alpha neuron pentru primul layer — captează dinamica temporală
        self.lif1 = neuron.ParametricLIFNode(
            init_tau=4.0,
            surrogate_function=sg,
            detach_reset=True
        )

        self.bn2     = nn.BatchNorm1d(out_dim)
        self.fc2     = layer.Linear(out_dim, out_dim, bias=False)
        self.dropout = layer.Dropout(p=dropout)
        # LIF simplu pentru al doilea — mai ieftin, suficient după filtrare
        self.lif2 = neuron.LIFNode(
            tau=2.0,
            surrogate_function=sg,
            detach_reset=True
        )

        self.shortcut = nn.Sequential(
            layer.Linear(in_dim, out_dim, bias=False),
            nn.BatchNorm1d(out_dim)
        ) if in_dim != out_dim else nn.Identity()
        
    def forward(self, x):
        identity = functional.seq_to_ann_forward(x, self.shortcut) \
                   if not isinstance(self.shortcut, nn.Identity) else x

        out = functional.seq_to_ann_forward(x, self.bn1)
        out = self.fc1(out)
        out = self.lif1(out)

        out = functional.seq_to_ann_forward(out, self.bn2)
        out = self.fc2(out)
        out = self.dropout(out)

        out = out + identity
        out = self.lif2(out)

        return out


class SNNFullyConnected(nn.Module):
    def __init__(self, input_dim: int = 4096, num_classes: int = 10, T: int = 32):
        super().__init__()
        self.T = T
        self.input_dim = input_dim

        # Blocuri FC cu ParametricLIF — fără threshold explicit
        self.block1 = FCAlphaBlock(input_dim, 512, dropout=0.5)
        self.block2 = FCAlphaBlock(512, 256, dropout=0.5)
        self.block3 = FCAlphaBlock(256, 128, dropout=0.4)

        self.classifier = nn.Sequential(
            nn.LayerNorm(128),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes)
        )

        functional.set_step_mode(self, step_mode='m')
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, layer.Linear)):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        functional.reset_net(self)

        spike_input = latency_encode(x, T=self.T)

        out = self.block1(spike_input)  # (T, B, 512)
        out = self.block2(out)          # (T, B, 256)
        out = self.block3(out)          # (T, B, 128)

        out = out.mean(0)               # (B, 128)

        return self.classifier(out)    # (B, num_classes)


MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_mels=N_MELS,
    n_fft=1024,
    hop_length=512,
).to(DEVICE)

DB_TRANSFORM  = torchaudio.transforms.AmplitudeToDB().to(DEVICE)
TIME_MASK     = T_aug.TimeMasking(time_mask_param=8)   # mai mic — spectrogram e deja mic
FREQ_MASK     = T_aug.FrequencyMasking(freq_mask_param=10)


def wav_to_flat_mel(wav: torch.Tensor, apply_augment: bool = False) -> torch.Tensor:
    mel = MEL_TRANSFORM(wav)   # (B, 1, N_MELS, T_frames) sau (1, N_MELS, T_frames)
    mel = DB_TRANSFORM(mel)

    # Redimensionăm la exact N_FRAMES pe axa temporală
    # interpolate asteaptă (B, C, H, W) sau (B, C, W)
    if mel.dim() == 3:
        mel = mel.unsqueeze(0)  # adăugăm dim batch dacă lipsește

    # mel shape: (B, 1, N_MELS, T_orig) → (B, 1, N_MELS, N_FRAMES)
    mel = F.interpolate(mel, size=(N_MELS, N_FRAMES), mode='bilinear', align_corners=False)

    if apply_augment:
        # Aplică augmentare pe fiecare sample din batch separat
        augmented = []
        for i in range(mel.shape[0]):
            s = mel[i]  # (1, N_MELS, N_FRAMES)
            if random.random() < 0.8:
                s = TIME_MASK(s)
            if random.random() < 0.8:
                s = FREQ_MASK(s)
            augmented.append(s)
        mel = torch.stack(augmented, dim=0)

    # Normalizare z-score per sample
    B = mel.shape[0]
    mel_flat = mel.view(B, -1)  # (B, N_MELS * N_FRAMES)
    mean = mel_flat.mean(dim=1, keepdim=True)
    std  = mel_flat.std(dim=1,  keepdim=True)
    mel_flat = (mel_flat - mean) / (std + 1e-6)

    return mel_flat  # (B, 2048)

class AudioDataset(Dataset):
    def __init__(self, df, audio_dir, target_sr=16000, duration=4.0, apply_augment=False):
        self.df            = df.reset_index(drop=True)
        self.audio_dir     = audio_dir
        self.target_sr     = target_sr
        self.target_len    = int(target_sr * duration)
        self.apply_augment = apply_augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        path     = os.path.join(self.audio_dir, f"fold{row['fold']}", row['slice_file_name'])
        wav, sr  = torchaudio.load(path)

        # Mono
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        # Resample
        if sr != self.target_sr:
            wav = torchaudio.functional.resample(wav, sr, self.target_sr)

        # Padding / crop la lungime fixă
        if wav.shape[1] < self.target_len:
            wav = F.pad(wav, (0, self.target_len - wav.shape[1]))
        else:
            wav = wav[:, :self.target_len]

        return wav, int(row['classID'])

class EarlyStopping:
    def __init__(self, patience=6, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_loss  = float("inf")
        self.counter    = 0
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_state = copy.deepcopy(model.state_dict())
            return False
        self.counter += 1
        return self.counter >= self.patience

def train_one_epoch(model, loader, optimizer, criterion, scheduler):
    model.train()
    total_loss = 0.0

    for wav, y in loader:
        wav, y = wav.to(DEVICE), y.to(DEVICE)
        x = wav_to_flat_mel(wav, apply_augment=True)

        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()

        # Gradient clipping mai strict pentru SNN FC (0.5 față de 1.0 la CNN)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.3)  # 0.5 → 0.3

        optimizer.step()
        scheduler.step() 
        total_loss += loss.item()

    return total_loss / len(loader)


def validate(model, loader, criterion):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for wav, y in loader:
            wav, y = wav.to(DEVICE), y.to(DEVICE)
            x      = wav_to_flat_mel(wav, apply_augment=False)
            logits = model(x)
            total_loss += criterion(logits, y).item()

    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for wav, y in loader:
            wav, y = wav.to(DEVICE), y.to(DEVICE)
            x      = wav_to_flat_mel(wav, apply_augment=False)
            preds  = torch.argmax(model(x), dim=1)
            correct += (preds == y).sum().item()
            total   += y.size(0)

    return correct / total

def run_cross_validation():
    df             = pd.read_csv(CSV_PATH)
    all_accuracies = []

    for fold in range(1, 11):
        print(f"\n{'='*40}")
        print(f"FOLD {fold}/10")
        print('='*40)

        train_full = df[df["fold"] != fold]
        test_df    = df[df["fold"] == fold]

        train_df, val_df = train_test_split(
            train_full,
            test_size=0.1,
            stratify=train_full["classID"],
            random_state=42
        )

        # Dataset-uri
        train_ds = AudioDataset(train_df, AUDIO_DIR, apply_augment=True)
        val_ds   = AudioDataset(val_df,   AUDIO_DIR, apply_augment=False)
        test_ds  = AudioDataset(test_df,  AUDIO_DIR, apply_augment=False)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=4, pin_memory=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=4, pin_memory=True)
        test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=4, pin_memory=True)

        # Model, optimizer, scheduler, criteriu
        model = SNNFullyConnected(
            input_dim=INPUT_DIM,
            num_classes=10,
            T=T_STEPS
        ).to(DEVICE)

        # Număr parametri
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Parametri: {n_params:,}")

        optimizer = torch.optim.AdamW(
        model.parameters(), lr=3e-4, weight_decay=5e-3  # LR mai mic, WD mai mare
        )

        # OneCycleLR e mai bun pentru SNN-uri FC — convergență mai rapidă
        steps_per_epoch = len(train_loader)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=10,        # restart la fiecare 10 epoci
        T_mult=2,      # perioadă dublă după fiecare restart
        eta_min=1e-6   # LR minim
        )
        # Label smoothing = 0.1 (mai mare decât la CNN — ajută la regularizare FC)
        criterion    = nn.CrossEntropyLoss(label_smoothing=0.15)
        early_stop = EarlyStopping(patience=8, min_delta=1e-3)  # patience 10→8, delta mai mare


        best_val_loss = float("inf")

        for epoch in range(1, EPOCHS + 1):
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scheduler)
            val_loss   = validate(model, val_loader, criterion)
            #scheduler.step()

            marker = " ← best" if val_loss < best_val_loss else ""
            if val_loss < best_val_loss:
                best_val_loss = val_loss

            print(f"Epoch {epoch:2d}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}{marker}")

            if early_stop.step(val_loss, model):
                print("Early stopping declanșat.")
                break

        # Restaurăm cel mai bun model
        model.load_state_dict(early_stop.best_state)

        acc = evaluate(model, test_loader)
        print(f"\nFold {fold} Accuracy: {acc:.4f} ({acc*100:.2f}%)")
        all_accuracies.append(acc)

    # ── Rezultate finale ────────────────────
    print("\n" + "="*40)
    print("REZULTATE FINALE — 10-Fold CV")
    print("="*40)
    for i, a in enumerate(all_accuracies, 1):
        print(f"  Fold {i:2d}: {a:.4f}")
    print(f"\n  Mean:  {np.mean(all_accuracies):.4f} ({np.mean(all_accuracies)*100:.2f}%)")
    print(f"  Std:   {np.std(all_accuracies):.4f}")
    print(f"  Min:   {np.min(all_accuracies):.4f}")
    print(f"  Max:   {np.max(all_accuracies):.4f}")

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].boxplot(all_accuracies, patch_artist=True,
                    boxprops=dict(facecolor='#9fa8da', color='#3949ab'),
                    medianprops=dict(color='#e53935', linewidth=2))
    axes[0].set_title("10-Fold CV Accuracy (SNN FC)")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_ylim(0, 1)
    axes[0].axhline(np.mean(all_accuracies), color='#e53935', linestyle='--',
                    label=f'Mean={np.mean(all_accuracies):.3f}')
    axes[0].legend()

    axes[1].bar(range(1, 11), all_accuracies, color='#9fa8da', edgecolor='#3949ab')
    axes[1].axhline(np.mean(all_accuracies), color='#e53935', linestyle='--',
                    label=f'Mean={np.mean(all_accuracies):.3f}')
    axes[1].set_xlabel("Fold")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title("Accuracy per fold")
    axes[1].set_ylim(0, 1)
    axes[1].legend()

    plt.tight_layout()
    plt.savefig("snn_fc_cv_results.png", dpi=150)
    plt.show()

    return all_accuracies

if __name__ == "__main__":
    # Decomentați quick_test_single_fold(1) pentru test rapid
    # quick_test_single_fold(1)

    run_cross_validation()

In [ ]:
"""
SNN Fully-Connected ( fara convolutii) cu RATE ENCODING pentru UrbanSound8K
=========================================================
Diferențe față de versiunea cu latency encoding:
  - rate_encode(): Bernoulli sampling — spike_prob = x_norm, T spikes independente
  - Nu mai există "un singur spike per neuron per T" — informația e distribuită
  - T_STEPS poate fi mai mic (16–20) față de latency (32) — rate e mai redundant
  - Suma spike-urilor pe T e mai "densă" → LIF tau mai mic (2.0 vs 4.0)
  - Dropout ușor redus — rate encoding regularizează implicit prin stochasticitate
"""

import os
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T_aug

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from spikingjelly.activation_based import neuron, functional, surrogate, layer

DATA_PATH  = "/kaggle/input/datasets/chrisfilo/urbansound8k"
CSV_PATH   = os.path.join(DATA_PATH, "UrbanSound8K.csv")
AUDIO_DIR  = DATA_PATH

# ── Hiperparametri ajustați pentru rate encoding ──────────────────────────────
# Rate encoding e mai puțin sensibil la T decât latency (informația e distribuită
# pe toți timestepii). T=20 e suficient; mai mult = mai mult zgomot Bernoulli.
T_STEPS   = 30       # ↓ față de 32 la latency (rate e redundant — nu ai nevoie de T mare)
N_MELS    = 64
N_FRAMES  = 64
INPUT_DIM = N_MELS * N_FRAMES   # 2048

BATCH_SIZE = 32
EPOCHS     = 40
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")
print(f"Input: {N_MELS}×{N_FRAMES} = {INPUT_DIM}, T={T_STEPS}")


# ── 1. RATE ENCODING (înlocuiește complet latency_encode) ────────────────────
def rate_encode(x: torch.Tensor, T: int = 20) -> torch.Tensor:
    """
    Rate encoding prin Bernoulli sampling.

    Args:
        x: (B, D) — features normalizate (după z-score, valori arbitrare)
        T: număr de timesteps

    Returns:
        spike_train: (T, B, D) — tensor binar float

    Mecanism:
        1. Normalizăm x în [0, 1] per-sample → probabilitatea de spike la fiecare timestep
        2. La fiecare timestep t, fiecare feature face un "coin flip" cu p = x_norm
        3. Rezultatul: features cu valori mari → spikes dese; mici → rare
        4. Spre deosebire de latency, FIECARE timestep e independent (fără memorie)
    """
    # Normalizare min-max per sample → [0, 1] = probabilitate de spike
    x_min = x.min(dim=1, keepdim=True)[0]   # (B, 1)
    x_max = x.max(dim=1, keepdim=True)[0]   # (B, 1)
    spike_prob = (x - x_min) / (x_max - x_min + 1e-6)  # (B, D) ∈ [0, 1]

    # Bernoulli sampling pentru fiecare timestep independent
    # spike_prob.unsqueeze(0) → (1, B, D), broadcast pe T
    spike_prob_exp = spike_prob.unsqueeze(0).expand(T, -1, -1)  # (T, B, D)
    spike_train = torch.bernoulli(spike_prob_exp)                # (T, B, D) ∈ {0, 1}

    return spike_train


# ── 2. BLOC FC cu LIF (tau mai mic pentru rate encoding) ─────────────────────
class FCRateBlock(nn.Module):
    """
    Bloc FC-LIF adaptat pentru rate encoding.

    Diferențe față de versiunea latency (FCAlphaBlock):
      - ParametricLIFNode cu init_tau=2.0 (↓ față de 4.0) — rate produce spikes dese,
        tau mic = integrare mai scurtă, neuronii nu se saturează
      - Al doilea LIF cu tau=1.5 — mai reactiv la densitatea de spikes
      - Dropout la 0.3 în loc de 0.5 — rate encoding are stochasticitate intrinsecă
        (Bernoulli sampling = regularizare implicită)
    """
    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.3):
        super().__init__()
        sg = surrogate.ATan(alpha=2.0)

        self.bn1  = nn.BatchNorm1d(in_dim)
        self.fc1  = layer.Linear(in_dim, out_dim, bias=False)
        self.lif1 = neuron.ParametricLIFNode(
            init_tau=2.0,           # ↓ față de 4.0 la latency — rate e dens
            surrogate_function=sg,
            detach_reset=True
        )

        self.bn2     = nn.BatchNorm1d(out_dim)
        self.fc2     = layer.Linear(out_dim, out_dim, bias=False)
        self.dropout = layer.Dropout(p=dropout)
        self.lif2    = neuron.LIFNode(
            tau=1.5,                # ↓ față de 2.0 — mai reactiv
            surrogate_function=sg,
            detach_reset=True
        )

        # Shortcut projection dacă dimensiunile diferă
        self.shortcut = nn.Sequential(
            layer.Linear(in_dim, out_dim, bias=False),
            nn.BatchNorm1d(out_dim)
        ) if in_dim != out_dim else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (T, B, in_dim)
        identity = functional.seq_to_ann_forward(x, self.shortcut) \
                   if not isinstance(self.shortcut, nn.Identity) else x

        out = functional.seq_to_ann_forward(x, self.bn1)
        out = self.fc1(out)
        out = self.lif1(out)          # (T, B, out_dim)

        out = functional.seq_to_ann_forward(out, self.bn2)
        out = self.fc2(out)
        out = self.dropout(out)

        out = out + identity          # skip connection
        out = self.lif2(out)

        return out                    # (T, B, out_dim)


class SNNRateEncoded(nn.Module):
    """
    SNN fully-connected cu rate encoding pentru UrbanSound8K.

    Pipeline:
        wav → mel spectrogram → z-score norm → rate_encode() →
        FCRateBlock×3 → mean over T → classifier → logits
    """
    def __init__(self, input_dim: int = 2048, num_classes: int = 10, T: int = 20):
        super().__init__()
        self.T = T

        self.block1 = FCRateBlock(input_dim, 512, dropout=0.3)
        self.block2 = FCRateBlock(512, 256, dropout=0.3)
        self.block3 = FCRateBlock(256, 128, dropout=0.25)

        self.classifier = nn.Sequential(
            nn.LayerNorm(128),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

        functional.set_step_mode(self, step_mode='m')
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, layer.Linear)):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        functional.reset_net(self)
        spike_input = rate_encode(x, T=self.T)   # (T, B, 2048)

        out = self.block1(spike_input)   # (T, B, 512)
        out = self.block2(out)           # (T, B, 256)
        out = self.block3(out)           # (T, B, 128)
        out = out.mean(dim=0)            # (B, 128)

        return self.classifier(out)      # (B, num_classes)


MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_mels=N_MELS,
    n_fft=1024,
    hop_length=512,
).to(DEVICE)

DB_TRANSFORM = torchaudio.transforms.AmplitudeToDB().to(DEVICE)
TIME_MASK    = T_aug.TimeMasking(time_mask_param=8)
FREQ_MASK    = T_aug.FrequencyMasking(freq_mask_param=10)


def wav_to_flat_mel(wav: torch.Tensor, apply_augment: bool = False) -> torch.Tensor:
    mel = MEL_TRANSFORM(wav)
    mel = DB_TRANSFORM(mel)

    if mel.dim() == 3:
        mel = mel.unsqueeze(0)

    mel = F.interpolate(mel, size=(N_MELS, N_FRAMES), mode='bilinear', align_corners=False)

    if apply_augment:
        augmented = []
        for i in range(mel.shape[0]):
            s = mel[i]
            if random.random() < 0.8:
                s = TIME_MASK(s)
            if random.random() < 0.8:
                s = FREQ_MASK(s)
            augmented.append(s)
        mel = torch.stack(augmented, dim=0)

    B = mel.shape[0]
    mel_flat = mel.view(B, -1)
    mean = mel_flat.mean(dim=1, keepdim=True)
    std  = mel_flat.std(dim=1, keepdim=True)
    mel_flat = (mel_flat - mean) / (std + 1e-6)

    return mel_flat   # (B, 2048)


class AudioDataset(Dataset):
    def __init__(self, df, audio_dir, target_sr=16000, duration=4.0, apply_augment=False):
        self.df            = df.reset_index(drop=True)
        self.audio_dir     = audio_dir
        self.target_sr     = target_sr
        self.target_len    = int(target_sr * duration)
        self.apply_augment = apply_augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        path    = os.path.join(self.audio_dir, f"fold{row['fold']}", row['slice_file_name'])
        wav, sr = torchaudio.load(path)

        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        if sr != self.target_sr:
            wav = torchaudio.functional.resample(wav, sr, self.target_sr)
        if wav.shape[1] < self.target_len:
            wav = F.pad(wav, (0, self.target_len - wav.shape[1]))
        else:
            wav = wav[:, :self.target_len]

        return wav, int(row['classID'])

class EarlyStopping:
    def __init__(self, patience=8, min_delta=1e-3):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_loss  = float("inf")
        self.counter    = 0
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_state = copy.deepcopy(model.state_dict())
            return False
        self.counter += 1
        return self.counter >= self.patience


def train_one_epoch(model, loader, optimizer, criterion, scheduler):
    model.train()
    total_loss = 0.0

    for wav, y in loader:
        wav, y = wav.to(DEVICE), y.to(DEVICE)
        x = wav_to_flat_mel(wav, apply_augment=True)

        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()

        # Gradient clipping — rate encoding are gradient mai stabil decât latency
        # (nu există "sparsitatea" extremă a unui singur spike per neuron)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)

        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def validate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for wav, y in loader:
            wav, y = wav.to(DEVICE), y.to(DEVICE)
            x      = wav_to_flat_mel(wav, apply_augment=False)
            total_loss += criterion(model(x), y).item()
    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for wav, y in loader:
            wav, y = wav.to(DEVICE), y.to(DEVICE)
            x      = wav_to_flat_mel(wav, apply_augment=False)
            preds  = torch.argmax(model(x), dim=1)
            correct += (preds == y).sum().item()
            total   += y.size(0)
    return correct / total


# ── 7. CROSS-VALIDATION 10-FOLD ───────────────────────────────────────────────
def run_cross_validation():
    df             = pd.read_csv(CSV_PATH)
    all_accuracies = []

    for fold in range(1, 11):
        print(f"\n{'='*40}")
        print(f"FOLD {fold}/10")
        print('='*40)

        train_full = df[df["fold"] != fold]
        test_df    = df[df["fold"] == fold]

        train_df, val_df = train_test_split(
            train_full,
            test_size=0.1,
            stratify=train_full["classID"],
            random_state=42
        )

        train_ds = AudioDataset(train_df, AUDIO_DIR, apply_augment=True)
        val_ds   = AudioDataset(val_df,   AUDIO_DIR, apply_augment=False)
        test_ds  = AudioDataset(test_df,  AUDIO_DIR, apply_augment=False)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=4, pin_memory=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=4, pin_memory=True)
        test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=4, pin_memory=True)

        model = SNNRateEncoded(
            input_dim=INPUT_DIM,
            num_classes=10,
            T=T_STEPS
        ).to(DEVICE)

        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Parametri: {n_params:,}")

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=3e-4,
            weight_decay=5e-3)

        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=10,
            T_mult=2,
            eta_min=1e-6
        )

        # Label smoothing 0.1 — mai mic față de latency (0.15) deoarece
        # rate encoding adaugă deja variabilitate prin Bernoulli sampling
        criterion  = nn.CrossEntropyLoss(label_smoothing=0.1)
        early_stop = EarlyStopping(patience=8, min_delta=1e-3)

        best_val_loss = float("inf")

        for epoch in range(1, EPOCHS + 1):
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scheduler)
            val_loss   = validate(model, val_loader, criterion)

            marker = " ← best" if val_loss < best_val_loss else ""
            if val_loss < best_val_loss:
                best_val_loss = val_loss

            print(f"Epoch {epoch:2d}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}{marker}")

            if early_stop.step(val_loss, model):
                print("Early stopping declanșat.")
                break

        model.load_state_dict(early_stop.best_state)

        acc = evaluate(model, test_loader)
        print(f"\nFold {fold} Accuracy: {acc:.4f} ({acc*100:.2f}%)")
        all_accuracies.append(acc)

    print("\n" + "="*40)
    print("REZULTATE FINALE — 10-Fold CV (Rate Encoding)")
    print("="*40)
    for i, a in enumerate(all_accuracies, 1):
        print(f"  Fold {i:2d}: {a:.4f}")
    print(f"\n  Mean:  {np.mean(all_accuracies):.4f} ({np.mean(all_accuracies)*100:.2f}%)")
    print(f"  Std:   {np.std(all_accuracies):.4f}")
    print(f"  Min:   {np.min(all_accuracies):.4f}")
    print(f"  Max:   {np.max(all_accuracies):.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].boxplot(all_accuracies, patch_artist=True,
                    boxprops=dict(facecolor='#b39ddb', color='#512da8'),
                    medianprops=dict(color='#e53935', linewidth=2))
    axes[0].set_title("10-Fold CV Accuracy (SNN Rate Encoding)")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_ylim(0, 1)
    axes[0].axhline(np.mean(all_accuracies), color='#e53935', linestyle='--',
                    label=f'Mean={np.mean(all_accuracies):.3f}')
    axes[0].legend()

    axes[1].bar(range(1, 11), all_accuracies, color='#b39ddb', edgecolor='#512da8')
    axes[1].axhline(np.mean(all_accuracies), color='#e53935', linestyle='--',
                    label=f'Mean={np.mean(all_accuracies):.3f}')
    axes[1].set_xlabel("Fold")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title("Accuracy per fold")
    axes[1].set_ylim(0, 1)
    axes[1].legend()

    plt.tight_layout()
    plt.savefig("snn_rate_cv_results.png", dpi=150)
    plt.show()

    return all_accuracies


if __name__ == "__main__":
    run_cross_validation()

In [2]:
pip install spikingjelly torch torchaudio scikit-learn pandas matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 8.1 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.
